# Temporal streams (temporaldata layer)

`piepy.temporal` turns a parsed session into a [temporaldata](https://github.com/neuro-galaxy/temporaldata) `Data`: the behavioural streams (wheel, licks, reward, stimulus windows) on **one clock**, built straight from the trial table's session-clock columns -- no rawdata.

Key ideas:
- The `trials` interval is the **domain** (and carries trial metadata).
- Streams that live on the rig clock (wheel/lick/reward) are shifted onto the **state** clock so events line up.
- A single **trial is just a slice** of the session `Data` -- there is no separate trial builder.

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs.

In [ ]:
import numpy as np
from temporaldata import Interval

from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.temporal import trial_slice
from piepy.tasks.wheel_detection.wheelDetectionSession import WheelDetectionSession
from piepy.tasks.wheel_detection.wheelDetectionStreams import WheelDetectionStreams

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Parse a single session
Builds the Session and stacks its runs onto one session clock (the `*_session` columns the stream builder reads).

In [ ]:
sess = WheelDetectionSession("240528_KC147_detect_opto120_PM__no_cam_KC")
data = sess.analyze(load_flag=True)

## Build the session streams
`WheelDetectionStreams` accepts a `Session` (it concatenates the runs for you) or an already-concatenated trial DataFrame. The base `SessionStreams` builds the universal `trials` domain; the wheel-detection subclass adds the rig-clock streams with the rig->state shift.

In [ ]:
data = WheelDetectionStreams(sess).build()
print("streams:", data.keys())
data

## The streams
All on the state session clock. Wheel carries a paired `position`; licks/reward are event times; `stim` is an interval.

In [ ]:
print("wheel  :", np.asarray(data.wheel.timestamps)[:5], "...  position", np.asarray(data.wheel.position)[:5])
print("licks  :", np.asarray(data.licks.timestamps)[:5])
print("reward :", np.asarray(data.reward.timestamps)[:5])
print("stim   :", np.asarray(data.stim.start)[:3], "->", np.asarray(data.stim.end)[:3])

## The `trials` domain
The trial intervals are the data's (gappy) domain and carry the trial-table metadata as attributes -- so "all hit trials" / "trial 5" is a selection, not a re-join.

In [ ]:
print("n trials :", len(np.asarray(data.trials.start)))
print("trial_no :", np.asarray(data.trials.trial_no)[:8])
print("outcome  :", np.asarray(data.trials.outcome)[:8])
print("start    :", np.asarray(data.trials.start)[:4])
print("end      :", np.asarray(data.trials.end)[:4])

## A trial is a slice (relative time)
`trial_slice` finds the trial's window and slices the session `Data`, returning every stream re-zeroed to the trial start. No separate trial builder.

In [ ]:
one = trial_slice(data, 5)
print("trial 5 wheel t:", np.asarray(one.wheel.timestamps)[:5])
print("trial 5 licks  :", np.asarray(one.licks.timestamps))
one

## Slice / align an arbitrary window
`Data.slice(start, end)` windows every stream and returns **relative** time -- handy for aligning to an event. `select_by_interval` keeps **absolute** time and can take a multi-segment interval (e.g. all hit trials at once).

In [ ]:
# a 2-second window, relative time out
win = data.slice(float(data.trials.start[0]), float(data.trials.start[0]) + 2000.0)
print("windowed wheel t:", np.asarray(win.wheel.timestamps)[:5])

# every hit trial, absolute time, as one multi-segment selection
trials = data.trials
hit = np.asarray(trials.outcome) == "hit"
hits = Interval(start=np.asarray(trials.start)[hit], end=np.asarray(trials.end)[hit])
hits_only = data.select_by_interval(hits)
print("hit-trial wheel samples:", len(np.asarray(hits_only.wheel.timestamps)))

## Extra streams for other tasks (declarative)
A task subclass exposes simple extra columns with a one-line `series` map: `"irregular"` reads a per-trial list of session-clock times (`<name>_session`); `"regular"` reads a column at `sampling_rate`. Anything needing pairing/clock-alignment goes in `extra_streams` (like wheel above). For example, a 2-photon task next to its Session/Trial:

```python
# tasks/my_2p_task/streams.py
from piepy.temporal import SessionStreams

class My2pStreams(SessionStreams):
    series = {"2p_frame": "regular"}          # frame times as a regular series
    sampling_rate = 30.0                       # Hz
    trial_attrs = ("trial_no", "outcome", "contrast")
    # override extra_streams(self, trials) for anything paired/aligned
```

## Interop / save
The result is a plain temporaldata `Data`, so it serializes to HDF5 and feeds any temporaldata-based pipeline (alignment, imaging, downstream ML).

In [ ]:
# data.to_hdf5("session_streams.h5")   # uncomment to persist
print("domain spans:", float(data.domain.start[0]), "->", float(data.domain.end[-1]), "ms")